In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
import optuna
import matplotlib.pyplot as plt
from datetime import datetime
from tabulate import tabulate

plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)

df = pd.read_csv('../data/외국인입국자_전처리완료_딥러닝용.csv')
df['성수기'] = df['월'].isin([7, 8, 12]).astype(int)
df['명절'] = df['월'].isin([1, 2, 9, 10]).astype(int)
df['코로나'] = (df['연도'] >= 2020).astype(int)

country = input("국적 입력 (없으면 Enter): ").strip()
purpose = input("목적 입력 (없으면 Enter): ").strip()

if country:
    df = df[df['국적'] == country]
else:
    print("🌍 국적 미선택 → 모든 나라 사용")

purpose_list = [purpose] if purpose else df['목적'].unique()
df['연월'] = df['연도'].astype(str) + '-' + df['월'].astype(str).str.zfill(2)

if not country:
    grouped = df.groupby(['연월', '목적', '성수기', '명절', '코로나'], as_index=False)['입국자수'].sum()
else:
    grouped = df.copy()

prophet_results = {}
xgb_results = {}
actuals = {}

def explain_score(r2, mape):
    if r2 >= 0.9:
        return "✅ 예측력 매우 우수 (신뢰도 높음)"
    elif r2 >= 0.7:
        return "👍 예측력 양호"
    elif r2 >= 0.5:
        return "⚠️ 예측력 보통 (개선 가능)"
    else:
        return "❗ 예측력 낮음 (참고용)"

def tune_xgb(X, y):
    def objective(trial):
        params = {
            'max_depth': trial.suggest_int('max_depth', 2, 6),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        }
        model = XGBRegressor(**params)
        X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, shuffle=False)
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)
        return np.mean((y_val - pred) ** 2)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=20, show_progress_bar=False)
    return study.best_params

for pur in purpose_list:
    temp = grouped[grouped['목적'] == pur].copy()
    if temp.empty:
        print(f"❌ 데이터 없음: {pur}")
        continue

    temp['ds'] = pd.to_datetime(temp['연월'])
    temp['y'] = temp['입국자수']

    # Prophet + 설명변수
    m = Prophet(yearly_seasonality=True)
    m.add_regressor('성수기')
    m.add_regressor('명절')
    m.add_regressor('코로나')
    m.fit(temp[['ds', 'y', '성수기', '명절', '코로나']])
    future = m.make_future_dataframe(periods=24, freq='M')
    future['월'] = future['ds'].dt.month
    future['성수기'] = future['월'].isin([7, 8, 12]).astype(int)
    future['명절'] = future['월'].isin([1, 2, 9, 10]).astype(int)
    future['코로나'] = (future['ds'].dt.year >= 2020).astype(int)
    forecast = m.predict(future)
    # ======= 마이너스 0으로 보정 =======
    forecast[['yhat', 'yhat_lower', 'yhat_upper']] = forecast[['yhat', 'yhat_lower', 'yhat_upper']].clip(lower=0)
    prophet_results[pur] = forecast

    # XGB (튜닝)
    temp['연도'] = temp['ds'].dt.year
    temp['월'] = temp['ds'].dt.month
    temp['lag_1'] = temp['y'].shift(1)
    temp['lag_3'] = temp['y'].shift(3)
    temp['lag_12'] = temp['y'].shift(12)
    temp = temp.dropna()
    features = ['연도', '월', 'lag_1', 'lag_3', 'lag_12', '성수기', '명절', '코로나']
    X = temp[features]
    y = temp['y']
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    best_params = tune_xgb(X_scaled, y)
    xgb = XGBRegressor(**best_params)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, shuffle=False, test_size=0.2)
    xgb.fit(X_train, y_train)
    y_pred = xgb.predict(X_test)

    xgb_df = pd.DataFrame({
        'ds': temp.iloc[-len(y_test):]['ds'].values,
        'xgb_yhat': y_pred
    })
    # ======= 마이너스 0으로 보정 =======
    xgb_df['xgb_yhat'] = xgb_df['xgb_yhat'].clip(lower=0)
    xgb_results[pur] = xgb_df
    actuals[pur] = temp

    # ========== 예측값 및 평가 해석 ==========
    real = temp['y'][-12:].values
    fcst = forecast['yhat'][:len(real)]  # 과거 구간 prophet 예측
    rmse_p = np.sqrt(mean_squared_error(real, fcst))
    r2_p = r2_score(real, fcst)
    mape_p = mean_absolute_percentage_error(real, fcst) * 100
    print(f"\n[{pur}] Prophet 예측 정확도 (최근 12개월)")
    print(f" - RMSE: {rmse_p:.1f} / MAPE: {mape_p:.2f}% / R²: {r2_p:.3f} → {explain_score(r2_p, mape_p)}")

    rmse_x = np.sqrt(mean_squared_error(y_test, y_pred))
    r2_x = r2_score(y_test, y_pred)
    mape_x = mean_absolute_percentage_error(y_test, y_pred) * 100
    print(f"[{pur}] XGBoost 예측 정확도 (테스트셋)")
    print(f" - RMSE: {rmse_x:.1f} / MAPE: {mape_x:.2f}% / R²: {r2_x:.3f} → {explain_score(r2_x, mape_x)}")

    last = forecast.iloc[-1]
    print(f"[{pur}] Prophet 미래 마지막달({last['ds'].strftime('%Y-%m')}) 예측값: {int(last['yhat']):,}명 (신뢰구간: {int(last['yhat_lower']):,} ~ {int(last['yhat_upper']):,})")

    plt.figure(figsize=(12,5))
    plt.plot(temp['ds'], temp['y'], label='실제값', color='black')
    plt.plot(forecast['ds'], forecast['yhat'], label='Prophet', color='red')
    plt.fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'],
                     alpha=0.2, color='pink', label='Prophet 신뢰구간')
    plt.plot(xgb_df['ds'], xgb_df['xgb_yhat'], label='XGBoost(튜닝)', color='blue')
    plt.title(f"[{pur}] Prophet + XGBoost(튜닝)")
    plt.legend()
    plt.show()

if not purpose:
    total_prophet = pd.concat(prophet_results.values()).groupby('ds').sum().reset_index()
    total_actuals = pd.concat(actuals.values()).groupby('ds').sum().reset_index()
    # ======= 마이너스 0으로 보정 =======
    total_prophet[['yhat', 'yhat_lower', 'yhat_upper']] = total_prophet[['yhat', 'yhat_lower', 'yhat_upper']].clip(lower=0)
    real = total_actuals['y'][-12:].values
    fcst = total_prophet['yhat'][:len(real)]
    rmse_p = np.sqrt(mean_squared_error(real, fcst))
    r2_p = r2_score(real, fcst)
    mape_p = mean_absolute_percentage_error(real, fcst) * 100
    print(f"\n[전체 목적] Prophet 예측 정확도 (최근 12개월)")
    print(f" - RMSE: {rmse_p:.1f} / MAPE: {mape_p:.2f}% / R²: {r2_p:.3f} → {explain_score(r2_p, mape_p)}")

    last = total_prophet.iloc[-1]
    print(f"[전체 목적] Prophet 미래 마지막달({last['ds'].strftime('%Y-%m')}) 예측값: {int(last['yhat']):,}명 (신뢰구간: {int(last['yhat_lower']):,} ~ {int(last['yhat_upper']):,})")

    plt.figure(figsize=(14,6))
    plt.plot(total_actuals['ds'], total_actuals['y'], label='실제값 합계', color='black')
    plt.plot(total_prophet['ds'], total_prophet['yhat'], label='Prophet 합계', color='red')
    plt.fill_between(total_prophet['ds'], total_prophet['yhat_lower'], total_prophet['yhat_upper'],
                     alpha=0.3, color='pink', label='Prophet 신뢰구간')
    plt.title("모든 목적 Prophet 합계")
    plt.legend()
    plt.show()

print("\n📅 원하는 연월 선택")
now = pd.to_datetime(datetime.now()).replace(day=1)
possible = pd.date_range(now, now + pd.DateOffset(months=24), freq='MS').strftime('%Y-%m').tolist()
rows = [possible[i:i+6] for i in range(0, len(possible), 6)]
print(tabulate(rows, tablefmt="grid"))

query = input("연월 입력 (202507 또는 2025-07): ").strip().replace('-', '')
if len(query) == 6:
    query = f"{query[:4]}-{query[4:]}"
query_input = query
query_year = int(query_input[:4])

if purpose:
    fc = prophet_results[purpose]
    fc[['yhat', 'yhat_lower', 'yhat_upper']] = fc[['yhat', 'yhat_lower', 'yhat_upper']].clip(lower=0)
    one_year = fc[fc['ds'].dt.year == query_year]
    mask = one_year['ds'].dt.strftime('%Y-%m') == query_input
    point = one_year[mask]
    plt.figure(figsize=(12,5))
    plt.plot(one_year['ds'], one_year['yhat'], label='Prophet', color='red')
    plt.fill_between(one_year['ds'], one_year['yhat_lower'], one_year['yhat_upper'],
                     alpha=0.3, color='pink')
    xgb = xgb_results[purpose]
    xgb_year = xgb[xgb['ds'].dt.year == query_year]
    plt.plot(xgb_year['ds'], xgb_year['xgb_yhat'], label='XGBoost(튜닝)', color='blue')
    if not point.empty:
        row = point.iloc[0]
        y_val = row['yhat']
        yhat_low = row['yhat_lower']
        yhat_up = row['yhat_upper']
        plt.scatter(row['ds'], y_val, color='black', label='선택달')
        plt.annotate(f"{int(y_val):,}", xy=(row['ds'], y_val),
                     xytext=(0,10), textcoords='offset points',
                     ha='center', fontsize=10, color='black')
        print(f"\n[{purpose}] {query_input} 예측: {int(y_val):,}명 (신뢰구간: {int(yhat_low):,} ~ {int(yhat_up):,})")
    else:
        # 가장 가까운 미래치라도 안내
        next_point = one_year[one_year['ds'] > pd.to_datetime(query_input + "-01")].head(1)
        if not next_point.empty:
            row = next_point.iloc[0]
            y_val = row['yhat']
            yhat_low = row['yhat_lower']
            yhat_up = row['yhat_upper']
            print(f"❗ 정확히 {query_input} 예측값은 없음. 가장 가까운 예측({row['ds'].strftime('%Y-%m')}): {int(y_val):,}명 (신뢰구간: {int(yhat_low):,} ~ {int(yhat_up):,})")
            plt.scatter(row['ds'], y_val, color='black', label='가까운 예측')
            plt.annotate(f"{int(y_val):,}", xy=(row['ds'], y_val),
                         xytext=(0,10), textcoords='offset points',
                         ha='center', fontsize=10, color='black')
        else:
            print(f"❗ {purpose} {query_input} 데이터 없음")
    plt.title(f"{purpose} Prophet+XGB ({query_year})")
    plt.legend()
    plt.show()

    # 전체 Prophet 합계
    total_prophet = pd.concat(prophet_results.values()).groupby('ds').sum().reset_index()
    total_prophet[['yhat', 'yhat_lower', 'yhat_upper']] = total_prophet[['yhat', 'yhat_lower', 'yhat_upper']].clip(lower=0)
    one_year_sum = total_prophet[total_prophet['ds'].dt.year == query_year]
    mask = one_year_sum['ds'].dt.strftime('%Y-%m') == query_input
    p2 = one_year_sum[mask]
    plt.figure(figsize=(12,5))
    plt.plot(one_year_sum['ds'], one_year_sum['yhat'], label='전체 Prophet', color='red')
    plt.fill_between(one_year_sum['ds'], one_year_sum['yhat_lower'], one_year_sum['yhat_upper'],
                     alpha=0.2, color='pink')
    if not p2.empty:
        row = p2.iloc[0]
        y_val = row['yhat']
        yhat_low = row['yhat_lower']
        yhat_up = row['yhat_upper']
        plt.scatter(row['ds'], y_val, color='black', label='선택달')
        plt.annotate(f"{int(y_val):,}", xy=(row['ds'], y_val),
                     xytext=(0,10), textcoords='offset points',
                     ha='center', fontsize=10, color='black')
        print(f"[모든 목적 합계] {query_input} 예측: {int(y_val):,}명 (신뢰구간: {int(yhat_low):,} ~ {int(yhat_up):,})")
    else:
        next_point = one_year_sum[one_year_sum['ds'] > pd.to_datetime(query_input + "-01")].head(1)
        if not next_point.empty:
            row = next_point.iloc[0]
            y_val = row['yhat']
            yhat_low = row['yhat_lower']
            yhat_up = row['yhat_upper']
            print(f"❗ 정확히 {query_input} 예측값은 없음. 가장 가까운 예측({row['ds'].strftime('%Y-%m')}): {int(y_val):,}명 (신뢰구간: {int(yhat_low):,} ~ {int(yhat_up):,})")
            plt.scatter(row['ds'], y_val, color='black', label='가까운 예측')
            plt.annotate(f"{int(y_val):,}", xy=(row['ds'], y_val),
                         xytext=(0,10), textcoords='offset points',
                         ha='center', fontsize=10, color='black')
        else:
            print(f"❗ 전체 목적 {query_input} 데이터 없음")
    plt.title(f"모든 목적 Prophet 합계 ({query_year})")
    plt.legend()
    plt.show()

if not purpose:
    one_year_sum = total_prophet[total_prophet['ds'].dt.year == query_year]
    one_year_sum[['yhat', 'yhat_lower', 'yhat_upper']] = one_year_sum[['yhat', 'yhat_lower', 'yhat_upper']].clip(lower=0)
    mask = one_year_sum['ds'].dt.strftime('%Y-%m') == query_input
    p2 = one_year_sum[mask]
    plt.figure(figsize=(12,5))
    plt.plot(one_year_sum['ds'], one_year_sum['yhat'], label='전체 Prophet', color='red')
    plt.fill_between(one_year_sum['ds'], one_year_sum['yhat_lower'], one_year_sum['yhat_upper'],
                     alpha=0.2, color='pink')
    if not p2.empty:
        row = p2.iloc[0]
        y_val = row['yhat']
        yhat_low = row['yhat_lower']
        yhat_up = row['yhat_upper']
        plt.scatter(row['ds'], y_val, color='black', label='선택달')
        plt.annotate(f"{int(y_val):,}", xy=(row['ds'], y_val),
                     xytext=(0,10), textcoords='offset points',
                     ha='center', fontsize=10, color='black')
        print(f"\n[모든 목적 합계] {query_input} 예측: {int(y_val):,}명 (신뢰구간: {int(yhat_low):,} ~ {int(yhat_up):,})")
    else:
        next_point = one_year_sum[one_year_sum['ds'] > pd.to_datetime(query_input + "-01")].head(1)
        if not next_point.empty:
            row = next_point.iloc[0]
            y_val = row['yhat']
            yhat_low = row['yhat_lower']
            yhat_up = row['yhat_upper']
            print(f"❗ 정확히 {query_input} 예측값은 없음. 가장 가까운 예측({row['ds'].strftime('%Y-%m')}): {int(y_val):,}명 (신뢰구간: {int(yhat_low):,} ~ {int(yhat_up):,})")
            plt.scatter(row['ds'], y_val, color='black', label='가까운 예측')
            plt.annotate(f"{int(y_val):,}", xy=(row['ds'], y_val),
                         xytext=(0,10), textcoords='offset points',
                         ha='center', fontsize=10, color='black')
        else:
            print(f"❗ 전체 목적 {query_input} 데이터 없음")
    plt.title(f"모든 목적 Prophet 합계 ({query_year})")
    plt.legend()
    plt.show()
